# Results Analysis

## Experiment 1 :  Fractals

### Model 1: Gemini-3.1-Pro 

#### Experiment's details

In [6]:
import pandas as pd
from IPython.display import display

# Definition of the configuration variables from your script
config_data = {
    "Variable Name": [
        "MODEL", 
        "TEMPERATURES", 
        "NUM_RUNS", 
        "DAT_BASELINE_RUNS",
        "INTER_CALL_SLEEP", 
        "RETRY_BACKOFF", 
        "IMAGE_DIRS",
        "RESULTS_DIR", 
        "BASELINE_OUTPUT", 
        "MAIN_OUTPUT"
    ],
    "Value in Code": [
        "google/gemini-3.1-pro-preview", 
        "[0.0, 1.0, 1.5]", 
        "10", 
        "10",
        "8", 
        "[30, 60, 120]", 
        "FD135, FD17",
        "results/Gemini-3.1-Pro", 
        "dat_baseline_results.jsonl", 
        "main_results.jsonl"
    ],
    "Simple Explanation": [
        "AI model name used for the experiment",
        "Creativity levels (0.0 = strict, 1.5 = highly creative)",
        "Number of repeated runs per image (Phase 2)",
        "Number of baseline runs (Phase 1)",
        "Seconds to wait between API calls (prevents server overload)",
        "Wait times in seconds if the server fails/disconnects",
        "Folders containing the input images",
        "Main folder where all results are saved",
        "Output file for Phase 1 results",
        "Output file for Phase 2 results"
    ]
}

# Create a Pandas DataFrame
df_config = pd.DataFrame(config_data)

# Apply some basic styling for better readability in Jupyter Notebook
styled_df = df_config.style.set_properties(**{'text-align': 'left'}).set_table_styles(
    [dict(selector='th', props=[('text-align', 'left'), ('font-weight', 'bold'), ('background-color', '#f4f4f4')])]
)

# Display the clean table
display(styled_df)

,Variable Name,Value in Code,Simple Explanation
0,MODEL,google/gemini-3.1-pro-preview,AI model name used for the experiment
1,TEMPERATURES,"[0.0, 1.0, 1.5]","Creativity levels (0.0 = strict, 1.5 = highly creative)"
2,NUM_RUNS,10,Number of repeated runs per image (Phase 2)
3,DAT_BASELINE_RUNS,10,Number of baseline runs (Phase 1)
4,INTER_CALL_SLEEP,8,Seconds to wait between API calls (prevents server overload)
5,RETRY_BACKOFF,"[30, 60, 120]",Wait times in seconds if the server fails/disconnects
6,IMAGE_DIRS,"FD135, FD17",Folders containing the input images
7,RESULTS_DIR,results/Gemini-3.1-Pro,Main folder where all results are saved
8,BASELINE_OUTPUT,dat_baseline_results.jsonl,Output file for Phase 1 results
9,MAIN_OUTPUT,main_results.jsonl,Output file for Phase 2 results


In [4]:

import json, math
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

pd.set_option('display.max_columns', 50)

# Lab plotting palette (kept consistent with other Anhedonic-AI / creativity figures)
COLOR_A = '#2b4c7e'   # dark blue
COLOR_B = '#c5d9ed'   # light blue
PALETTE = ['#2b4c7e', '#c5d9ed', '#e0a458', '#7ea172', '#a35b6b', '#8c7ab3']

plt.rcParams.update({
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.linewidth': 1.2,
    'figure.dpi': 110,
})


In [9]:

def load_model_jsonl(path, model_name):
    rows = []
    with open(path) as f:
        for line in f:
            r = json.loads(line)
            fp = r.get('fractal_parsed')
            percepts = fp.get('percepts') if fp else None
            parse_ok = percepts is not None

            n_percepts = len(percepts) if parse_ok else np.nan
            n_face = (sum(1 for p in percepts if 'face' in p.get('label', '').lower())
                      if parse_ok else np.nan)
            mean_conf = (np.mean([p.get('confidence_present', np.nan) for p in percepts])
                         if parse_ok and percepts else np.nan)

            rows.append(dict(
                model=model_name,
                file=r['file'],
                fd=r['fd'],
                fd_value=1.35 if r['fd'] == 'FD135' else 1.7,
                temp_label=r['temp_label'],
                temperature=r['temperature'],
                run_id=r['run_id'],
                parse_ok=parse_ok,
                n_percepts=n_percepts,
                n_face_percepts=n_face,
                mean_confidence=mean_conf,
                percepts=percepts,
                dat_words=[w.lower() for w in r['dat_parsed']['words']],
                ms_fractal=r.get('ms_fractal'),
                ms_dat=r.get('ms_dat'),
            ))
    return pd.DataFrame(rows)


# --- Add every model's export here (same schema) ---
MODEL_FILES = {
    "Gemini 3.1 Pro": "/Users/precioux/Desktop/Projects/Creativity/code/Server/Fractals/results/Gemini-3.1-Pro/main_results.jsonl",
    # "Qwen3.5-9B":        "qwen_fractal_results.jsonl",
    # "Claude Opus 4.7":   "opus_fractal_results.jsonl",
    # "Mistral Small":     "mistral_fractal_results.jsonl",
    # "GPT-5.4":           "gpt5_fractal_results.jsonl",
    # "Gemma 4 26B":       "gemma_fractal_results.jsonl",
}

df = pd.concat([load_model_jsonl(p, name) for name, p in MODEL_FILES.items()], ignore_index=True)
print(f"{len(df)} trials loaded across {df['model'].nunique()} model(s):", list(MODEL_FILES))
df.head(3)


4525 trials loaded across 1 model(s): ['Gemini 3.1 Pro']


,model,file,fd,fd_value,temp_label,temperature,run_id,parse_ok,n_percepts,n_face_percepts,mean_confidence,percepts,dat_words,ms_fractal,ms_dat
0,Gemini 3.1 Pro,FD_1.35_0.png,FD135,1.35,deterministic,0.0,1,True,4.0,1.0,67.50,"[{'label': 'face', 'description': 'A profile o...","[justice, elephant, galaxy, sorrow, screwdrive...",5106,11013
1,Gemini 3.1 Pro,FD_1.35_0.png,FD135,1.35,deterministic,0.0,2,True,4.0,1.0,76.25,"[{'label': 'face', 'description': 'A profile o...","[galaxy, sorrow, moss, hammer, democracy, soup...",4124,9337
2,Gemini 3.1 Pro,FD_1.35_0.png,FD135,1.35,deterministic,0.0,3,True,4.0,1.0,76.25,"[{'label': 'face', 'description': 'A profile o...","[ocean, philosophy, reptile, engine, symphony,...",5998,18045



#### 2. Data quality: parse failures & degenerate generations

Two failure modes worth flagging *before* trusting any percept-count statistic:

1. **Parse failures** — `fractal_parsed is None` because the model's JSON didn't come back clean.
2. **Degenerate repetition loops** — a small fraction of runs report a huge, clearly-broken number of
   percepts (e.g. hundreds, capped near 100, or one case at 1585). These are decoding pathologies
   (the model loops or hits a max-token wall), not genuine creativity, and would badly distort a mean
   if left in. We flag anything above `DEGENERATE_THRESHOLD` percepts and exclude it from percept-count
   analyses (kept in a separate `degenerate` flag so you can inspect it).


In [11]:

DEGENERATE_THRESHOLD = 30  # percept counts above this are decoding pathologies, not creativity

df['degenerate'] = df['n_percepts'] > DEGENERATE_THRESHOLD

quality = (df.groupby('model')
             .agg(n_trials=('file', 'size'),
                  parse_failures=('parse_ok', lambda s: (~s).sum()),
                  degenerate_runs=('degenerate', 'sum'))
             .assign(parse_failure_rate=lambda d: (d['parse_failures'] / d['n_trials']).round(4),
                     degenerate_rate=lambda d: (d['degenerate_runs'] / d['n_trials']).round(4)))
quality


,n_trials,parse_failures,degenerate_runs,parse_failure_rate,degenerate_rate
model,,,,,
Gemini 3.1 Pro,4525,14,103,0.0031,0.0228


In [12]:

# Where do degenerate runs cluster? (useful for a methods note / appendix)
df[df['degenerate']].groupby(['model', 'temp_label']).size().rename('n_degenerate_runs')


model           temp_label   
Gemini 3.1 Pro  default          53
                high-creative    50
Name: n_degenerate_runs, dtype: int64